In [ ]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import nltk
import re

In [ ]:
input_file = 'Dataset.txt'
with open(input_file, 'r', encoding='utf-8') as infile:
    data = infile.read()

In [ ]:

def preprocess_pipeline(data) -> 'list':
    sentences = data.split('\n')
    sentences = [s.strip() for s in sentences]
    sentences = [s for s in sentences if len(s) > 0]
    # Tokenization
    tokenized = []
    for sentence in sentences:
        # Convert to lowercase
        sentence = sentence.lower()
        tokenized.append(sentence)
    return tokenized

# Tokenize sentences
tokenized_sentences = preprocess_pipeline(data)

In [ ]:
tokenizer = Tokenizer(oov_token='<oov>')
tokenizer.fit_on_texts(tokenized_sentences)
total_words = len(tokenizer.word_index) + 1

input_sequences = []
for line in tokenized_sentences:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i + 1]
        input_sequences.append(n_gram_sequence)
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

In [ ]:
X,labels = input_sequences[:,:-1],input_sequences[:,-1]
ys = tf.keras.utils.to_categorical(labels, num_classes=total_words)

In [ ]:
from sklearn.model_selection import train_test_split
X_train_temp, X_val_test, y_train_temp, y_val_test = train_test_split(X, ys, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_val_test, y_val_test, test_size=0.5, random_state=42)

In [ ]:
model = Sequential()
model.add(Embedding(total_words, 150))
model.add(Bidirectional(LSTM(200)))
model.add(Dense(total_words, activation='softmax'))
adam = Adam(learning_rate=0.01)
model.compile(loss='categorical_crossentropy', optimizer=adam, metrics=['accuracy'])
history = model.fit(X_train_temp, y_train_temp, epochs=15, validation_data=(X_val, y_val), verbose=1)

Epoch 1/15
38/38 ━━━━━━━━━━━━━━━━━━━━ 12s 166ms/step - accuracy: 0.0344 - loss: 6.3920 - val_accuracy: 0.0795 - val_loss: 5.9713
Epoch 2/15
38/38 ━━━━━━━━━━━━━━━━━━━━ 6s 153ms/step - accuracy: 0.0776 - loss: 5.4405 - val_accuracy: 0.0795 - val_loss: 6.1921
Epoch 3/15
38/38 ━━━━━━━━━━━━━━━━━━━━ 4s 118ms/step - accuracy: 0.0969 - loss: 4.9728 - val_accuracy: 0.1060 - val_loss: 6.3455
Epoch 4/15
38/38 ━━━━━━━━━━━━━━━━━━━━ 4s 105ms/step - accuracy: 0.1822 - loss: 4.2979 - val_accuracy: 0.1258 - val_loss: 6.5416
Epoch 5/15
38/38 ━━━━━━━━━━━━━━━━━━━━ 6s 140ms/step - accuracy: 0.2348 - loss: 3.5940 - val_accuracy: 0.1391 - val_loss: 6.9553
Epoch 6/15
38/38 ━━━━━━━━━━━━━━━━━━━━ 9s 116ms/step - accuracy: 0.3961 - loss: 2.6033 - val_accuracy: 0.1126 - val_loss: 7.5612
Epoch 7/15
38/38 ━━━━━━━━━━━━━━━━━━━━ 4s 89ms/step - accuracy: 0.6056 - loss: 1.7777 - val_accuracy: 0.1325 - val_loss: 8.0163
Epoch 8/15
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 84ms/step - accuracy: 0.7694 - loss: 1.0288 - val_accuracy: 0.1

In [ ]:
def predict_top_five_words(model, tokenizer, seed_text):
    token_list = tokenizer.texts_to_sequences([seed_text])[0]
    token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
    predicted = model.predict(token_list, verbose=0)
    top_five_indexes = np.argsort(predicted[0])[-5:][::-1]
    index_to_word_map = {idx: word for word, idx in tokenizer.word_index.items()}
    top_five_words = [index_to_word_map.get(index, "[UNK]") for index in top_five_indexes]
    return top_five_words

In [ ]:
from IPython.display import display, HTML
def predict_and_display_top_five_words(seed_text, model, tokenizer):
    top_five_words = predict_top_five_words(model, tokenizer, seed_text)
    html_items = ""
    for word in top_five_words:
        html_items += f"<li>{seed_text} <strong>{word}</strong></li>"
    output_html = f"""
    <div style="font-family: Arial, sans-serif; border: 1px solid #e1e4e8; border-radius: 6px; padding: 16px;">
        <h4 style="margin-top:0; margin-bottom:12px;">Sentence Autocompletion App ✍️</h4>
        <p style="color: #586069;">Five possible outputs for "<em>{seed_text}</em>":</p>
        <ul style="margin-bottom:0;">{html_items}</ul>
    </div>
    """
    display(HTML(output_html))

In [ ]:
model.save('next_word_model.keras')

In [ ]:
from tensorflow.keras.models import load_model
loaded_model = load_model('next_word_model.keras')
from tensorflow.keras.preprocessing.text import Tokenizer
loaded_tokenizer = Tokenizer(oov_token='<oov>')
max_sequence_len = 16

In [ ]:
loaded_tokenizer.fit_on_texts(tokenized_sentences)

In [ ]:
from flask import Flask, request, jsonify

app = Flask(__name__)

In [ ]:
@app.route('/predict', methods=['POST'])
def predict():
    data = request.get_json(force=True)
    seed_text = data['text']
    top_five_words = predict_top_five_words(loaded_model, loaded_tokenizer, seed_text)

    return jsonify({'predictions': top_five_words})

In [ ]:
def predict_top_five_words(model, tokenizer, seed_text, num_words_to_generate=5):
    generated_sequence = seed_text
    for _ in range(num_words_to_generate):
        token_list = tokenizer.texts_to_sequences([generated_sequence])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len - 1, padding='pre')
        predicted_probabilities = model.predict(token_list, verbose=0)
        predicted_word_index = np.argmax(predicted_probabilities)
        if predicted_word_index == 0:
            predicted_word = tokenizer.index_word.get(predicted_word_index, "")
        else:
            predicted_word = tokenizer.index_word.get(predicted_word_index, "")

        if predicted_word:
            generated_sequence += " " + predicted_word
        else:
            break
    return generated_sequence

In [ ]:
def predict_top_five_words(model, tokenizer, seed_text, num_suggestions=5, num_words_per_suggestion=10, top_n=3):
    sequences = [(seed_text, 1.0)]  # Store (generated_text, probability) tuples
    all_suggestions = []

    for _ in range(num_words_per_suggestion):
        new_sequences = []
        for current_text, current_prob in sequences:
            if len(all_suggestions) >= num_suggestions:
                break

            token_list = tokenizer.texts_to_sequences([current_text])[0]
            token_list = pad_sequences([token_list], maxlen=max_sequence_len - 1, padding='pre')
            predicted_probabilities = model.predict(token_list, verbose=0)[0]

            top_n_indices = np.argsort(predicted_probabilities)[-top_n:][::-1]

            for index in top_n_indices:
                predicted_word = tokenizer.index_word.get(index, "")
                if predicted_word and predicted_word != '<oov>':
                    new_text = current_text + " " + predicted_word
                    new_prob = current_prob * predicted_probabilities[index]
                    new_sequences.append((new_text, new_prob))

        sequences = sorted(new_sequences, key=lambda x: x[1], reverse=True)[:num_suggestions]

        # Add completed sequences to all_suggestions if they reach a certain length or condition
        for seq, prob in sequences:
            if seq not in [s for s, p in all_suggestions]: # Avoid duplicates
                 all_suggestions.append((seq, prob))

        if len(all_suggestions) >= num_suggestions:
            break

    # Sort by probability and take the top num_suggestions distinct sentences
    all_suggestions = sorted(list(set([s for s,p in all_suggestions])), key=lambda x: -sequences[[s for s,p in sequences].index(x)][1] if x in [s for s,p in sequences] else -1)[:num_suggestions]

    return all_suggestions

In [ ]:
def handle_input(change):
    input_text = change.new
    if not input_text:
        output_area.layout.display = 'none'
        return
    suggestions = predict_top_five_words(model, tokenizer, input_text)

    if suggestions:
        output_area.options = suggestions
        output_area.layout.display = 'block'
    else:
        output_area.layout.display = 'none'
text_input.observe(handle_input, names='value')

In [ ]:
def on_suggestion_select(change):
    if not change.new:
        return
    text_input.unobserve(handle_input, names='value')
    selected_sentence = change.new
    text_input.value = selected_sentence + " "
    output_area.layout.display = 'none'
    text_input.observe(handle_input, names='value')

In [ ]:
def predict_top_five_words_beam_search(model, tokenizer, seed_text, num_suggestions=5, num_words_per_suggestion=10, beam_width=5):
    sequences = [(tokenizer.texts_to_sequences([seed_text])[0], 0.0)]
    all_suggestions = []

    for _ in range(num_words_per_suggestion):
        new_sequences = []
        for current_sequence, current_log_prob in sequences:
            if len(all_suggestions) >= num_suggestions:
                break

            token_list = pad_sequences([current_sequence], maxlen=max_sequence_len - 1, padding='pre')
            predicted_probabilities = model.predict(token_list, verbose=0)[0]
            top_beam_width_indices = np.argsort(predicted_probabilities)[-beam_width:][::-1]

            for index in top_beam_width_indices:
                predicted_word = tokenizer.index_word.get(index, "")
                if predicted_word and predicted_word != '<oov>':
                    new_sequence = current_sequence + [index]
                    new_log_prob = current_log_prob + np.log(predicted_probabilities[index])
                    new_sequences.append((new_sequence, new_log_prob))

        sequences = sorted(new_sequences, key=lambda x: x[1], reverse=True)[:beam_width]
    for seq, prob in sequences:
        text_suggestion = tokenizer.sequences_to_texts([seq])[0]
        all_suggestions.append(text_suggestion)

    return all_suggestions[:num_suggestions]

In [ ]:
def handle_input(change):
    input_text = change.new
    if not input_text:
        output_area.layout.display = 'none'
        return
    suggestions = predict_top_five_words_beam_search(model, tokenizer, input_text)

    if suggestions:
        output_area.options = suggestions
        output_area.layout.display = 'block'
    else:
        output_area.layout.display = 'none'

text_input.observe(handle_input, names='value')

In [ ]:
model_path = 'next_word_model.keras'
model = tf.keras.models.load_model(model_path)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.experimental_enable_resource_variables = True
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
converter._experimental_lower_tensor_list_ops = False
tflite_model = converter.convert()
quantized_model_path = 'next_word_model_quantized.tflite'
with open(quantized_model_path, 'wb') as f:
    f.write(tflite_model)

print(f"Quantized model saved to: {quantized_model_path}")

Saved artifact at '/tmp/tmp9voos6g3'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 15), dtype=tf.float32, name='input_layer_8')
Output Type:
  TensorSpec(shape=(None, 727), dtype=tf.float32, name=None)
Captures:
  140633673637264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140633673642640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140633673643408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140633673643984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140633673641104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140633673641872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140633673642448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140633673643024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140633673644368: TensorSpec(shape=(), dtype=tf.resource, name=None)
Quantized model saved to: next_word_model_quantized.tflite


In [ ]:
interpreter = tf.lite.Interpreter(model_path=quantized_model_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input details:", input_details)
print("Output details:", output_details)

def predict_with_tflite(interpreter, input_text, tokenizer, max_sequence_len):
    token_list = tokenizer.texts_to_sequences([input_text])[0]
    padded_sequence = pad_sequences([token_list], maxlen=max_sequence_len - 1, padding='pre')
    input_dtype = input_details[0]['dtype']
    if padded_sequence.dtype != input_dtype:
        padded_sequence = padded_sequence.astype(input_dtype)

    interpreter.set_tensor(input_details[0]['index'], padded_sequence)
    interpreter.invoke()

    output_data = interpreter.get_tensor(output_details[0]['index'])
    predicted_probabilities = output_data[0]
    return predicted_probabilities

Input details: [{'name': 'serving_default_input_layer_8:0', 'index': 0, 'shape': array([ 1, 15], dtype=int32), 'shape_signature': array([-1, 15], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
Output details: [{'name': 'StatefulPartitionedCall_1:0', 'index': 48, 'shape': array([  1, 727], dtype=int32), 'shape_signature': array([ -1, 727], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]


In [ ]:
def predict_top_five_words_beam_search_tflite(interpreter, tokenizer, seed_text, max_sequence_len, num_suggestions=5, num_words_per_suggestion=10, beam_width=5):
    sequences = [(tokenizer.texts_to_sequences([seed_text])[0], 0.0)]
    all_suggestions = []
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    for _ in range(num_words_per_suggestion):
        new_sequences = []
        for current_sequence, current_log_prob in sequences:
            if len(all_suggestions) >= num_suggestions:
                break
            token_list = pad_sequences([current_sequence], maxlen=max_sequence_len - 1, padding='pre')
            input_dtype = input_details[0]['dtype']
            if token_list.dtype != input_dtype:
                 token_list = token_list.astype(input_dtype)

            interpreter.set_tensor(input_details[0]['index'], token_list)
            interpreter.invoke()
            predicted_output = interpreter.get_tensor(output_details[0]['index'])[0]
            predicted_probabilities = predicted_output
            top_beam_width_indices = np.argsort(predicted_probabilities)[-beam_width:][::-1]

            for index in top_beam_width_indices:
                predicted_word = tokenizer.index_word.get(index, "")
                if predicted_word and predicted_word != '<oov>':
                    new_sequence = current_sequence + [index]
                    prob = predicted_probabilities[index]
                    new_log_prob = current_log_prob + (np.log(prob) if prob > 0 else -np.inf)
                    new_sequences.append((new_sequence, new_log_prob))

        sequences = sorted(new_sequences, key=lambda x: x[1], reverse=True)[:beam_width]
    for seq, prob in sequences:
        text_suggestion = tokenizer.sequences_to_texts([seq])[0]
        all_suggestions.append(text_suggestion)
    return all_suggestions[:num_suggestions]

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import time
import threading

text_input = widgets.Text(
    description="Enter text:",
    placeholder="Type here..."
)

output_area = widgets.Select(
    options=[],
    rows=5,
    description='',
    disabled=False
)
output_area.layout.display = 'none'

debounce_delay = 0.1
debounce_timer = None

def handle_input(change):
    global debounce_timer
    input_text = change.new
    if input_text.strip().endswith('.'):
        output_area.options = []
        output_area.layout.display = 'none'
        if debounce_timer and debounce_timer.is_alive():
            debounce_timer.cancel()
        return
    if not input_text.strip():
        output_area.options = []
        output_area.layout.display = 'none'
        if debounce_timer and debounce_timer.is_alive():
            debounce_timer.cancel()
        return
    if debounce_timer and debounce_timer.is_alive():
        debounce_timer.cancel()
    debounce_timer = threading.Timer(debounce_delay, predict_and_display, args=[input_text.strip()])
    debounce_timer.start()

def predict_and_display(input_text):
    start_time = time.time()
    suggestions = predict_top_five_words_beam_search_tflite(interpreter, tokenizer, input_text, max_sequence_len)
    end_time = time.time()
    prediction_duration = end_time - start_time
    print(f"Prediction time: {prediction_duration:.4f} seconds")

    all_suggestions = []
    for suggestion in suggestions:
        if suggestion.lower().startswith(input_text.lower()):
            completion = suggestion[len(input_text):].strip()
            if completion and completion not in all_suggestions:
                all_suggestions.append(completion)
            words = completion.split()
            for i in range(1, len(words)):
                prefix = " ".join(words[:i])
                if prefix and prefix not in all_suggestions:
                     all_suggestions.append(prefix)
    all_suggestions = list(dict.fromkeys(all_suggestions))[:10]

    if all_suggestions:
        output_area.options = all_suggestions
        output_area.layout.display = 'block'
    else:
        output_area.layout.display = 'none'


def on_suggestion_select(change):
    if not change.new:
        return
    text_input.unobserve(handle_input, names='value')
    selected_completion = change.new
    current_text = text_input.value.strip()
    text_input.value = f"{current_text} {selected_completion} "
    output_area.layout.display = 'none'
    text_input.observe(handle_input, names='value')
text_input.observe(handle_input, names='value')
output_area.observe(on_suggestion_select, names='value')
app_layout = widgets.VBox([text_input, output_area])
display(app_layout)